# dARK Lima two-site: network and deposit acceptance

This notebook accepts the five-VM two-site Lima laboratory. It proves that same-site traffic uses `site-a-lan`/`site-b-lan`, cross-site Besu and IPFS traffic uses `mesh-vpn`, WireGuard is present on every guest, storage APIs are reachable on the required networks, and the public ARK lifecycle still works.

Run it only after `lima-two-site-up.sh`, inventory generation, `install`, and `verify`. The test creates disposable authority and ARK data.

In [ ]:
from __future__ import annotations

import json, os, shlex, subprocess, sys, time, uuid
from pathlib import Path
import requests

ROOT = Path(os.getenv('DARK_DEPLOYER_ROOT', Path.cwd())).expanduser().resolve()
if not (ROOT / 'deploy.py').exists():
    raise RuntimeError('Set DARK_DEPLOYER_ROOT to the dark-deployer checkout')
sys.path.insert(0, str(ROOT))
INVENTORY = ROOT / os.getenv('DARK_INVENTORY', 'examples/operator-inventory/lima-two-site-five-host.json')
GATEWAY_BASE_URL = os.getenv('DARK_GATEWAY_BASE_URL', 'http://192.168.105.15').rstrip('/')
AUTHORITY_ID = os.getenv('AUTHORITY_ID', f'two-site-notebook-{uuid.uuid4().hex[:12]}')
NAAN = os.getenv('NAAN', '12345')
PLATFORM_ITEM_ID = os.getenv('PLATFORM_ITEM_ID', f'two-site-item-{uuid.uuid4().hex[:12]}')
TARGET_URL = os.getenv('TARGET_URL', f'https://repository.example.org/items/{PLATFORM_ITEM_ID}')
POLL_TIMEOUT_SECONDS = int(os.getenv('POLL_TIMEOUT_SECONDS', '300'))
print('inventory:', INVENTORY)
print('gateway:', GATEWAY_BASE_URL)

## 1. Validate the topology and selected networks

In [ ]:
from deployment_v3.inventory_resolver import resolve_inventory_path
from deployment_v3.planner import build_plan
from deployment_v3.network import internal_port

resolution = resolve_inventory_path(INVENTORY)
plan = build_plan(INVENTORY)
assert resolution.document['deployment']['id'] == 'dark-lima-two-site-five-host'
assert set(resolution.document['networks']) == {'site-a-lan', 'site-b-lan', 'mesh-vpn'}
assert len(plan.machines) == 5

def edges(family):
    return plan.raw['peerings'][family]

assert all(edge['network'] == 'mesh-vpn' for edge in edges('ipfs'))
assert all(edge['network'] == 'mesh-vpn' for edge in edges('cluster'))
blockchain_local = next(edge for edge in edges('blockchain') if edge['from'] == 'validator01' and edge['to'] == 'validator02')
blockchain_remote = next(edge for edge in edges('blockchain') if edge['from'] == 'validator01' and edge['to'] == 'validator03')
assert blockchain_local['network'] == 'site-a-lan'
assert blockchain_remote['network'] == 'mesh-vpn'
store = plan.service('store-api')
assert store.connections['cluster_cluster-storage-a']['network'] == 'site-a-lan'
assert store.connections['cluster_cluster-storage-b']['network'] == 'mesh-vpn'
for service_id in ('ipfs-storage-a', 'cluster-storage-a'):
    service = plan.service(service_id)
    assert {item['network'] for item in service.listeners} == {'site-a-lan', 'mesh-vpn'}
assert plan.service('ipfs-storage-b').exposure['network'] == 'mesh-vpn'
assert plan.service('cluster-storage-b').exposure['network'] == 'mesh-vpn'
print('Topology and locality routing passed')

## 2. Verify WireGuard, Docker, and storage runtime

In [ ]:
def ssh_command(machine_id):
    machine = resolution.document['machines'][machine_id]
    ssh = machine['ssh']
    return ['ssh', '-o', 'BatchMode=yes', '-o', 'StrictHostKeyChecking=yes', '-o', f"UserKnownHostsFile={ssh['known_hosts_file']}", '-i', ssh['private_key_file'], '-p', str(ssh['port']), f"{ssh['user']}@{machine['management_address']}"]

def ssh_run(machine_id, command, **kwargs):
    return subprocess.run([*ssh_command(machine_id), f"sh -lc {shlex.quote(command)}"], text=True, capture_output=True, **kwargs)

for machine in plan.machines:
    result = ssh_run(machine.id, "ip -4 addr show dev wg0; docker info --format '{{.SecurityOptions}}'")
    assert result.returncode == 0, (machine.id, result.stderr)
    assert '10.250.30.' in result.stdout, (machine.id, result.stdout)
    assert 'rootless' not in result.stdout.lower(), (machine.id, result.stdout)

for service_id in ('ipfs-storage-a', 'ipfs-storage-b', 'cluster-storage-a', 'cluster-storage-b'):
    service = plan.service(service_id)
    machine = plan.machine(service.machine_id)
    command = ['docker', 'ps', '--filter', f'label=org.dark.service.id={service_id}', '--filter', 'status=running', '--format', '{{.Names}}']
    result = ssh_run(machine.id, shlex.join(command))
    assert result.returncode == 0 and len(result.stdout.splitlines()) == 1, (service_id, result.stdout, result.stderr)

print('WireGuard, rootful Docker, Kubo, and Cluster runtime checks passed')

## 3. Check the public gateway

In [ ]:
health = requests.get(f'{GATEWAY_BASE_URL}/health', timeout=30)
assert health.status_code < 500, health.text
workers = requests.get(f'{GATEWAY_BASE_URL}/api/v1/worker/status', timeout=30)
workers.raise_for_status()
for worker in ('metadata', 'replication', 'chain'):
    assert workers.json()['workers'][worker]['alive'] is True
assert requests.get(f'{GATEWAY_BASE_URL}/api/docs', timeout=30).status_code < 500
assert requests.get(f'{GATEWAY_BASE_URL}/api/openapi.json', timeout=30).status_code == 200
print('Gateway, workers, and Minter documentation passed')

## 4. Create, replicate, and resolve an ARK

In [ ]:
MINTER = f'{GATEWAY_BASE_URL}/api/v1'
HEADERS = {'X-Authority-Id': AUTHORITY_ID}
def private_request(service_id, method, path, payload=None):
    service = plan.service(service_id)
    container = ssh_run(service.machine_id, shlex.join(['docker', 'ps', '-q', '--filter', f'label=org.dark.service.id={service_id}', '--filter', 'status=running']), check=True).stdout.strip().splitlines()[0]
    request = {'method': method, 'path': path, 'payload': payload}
    port = internal_port(service)
    script = "import json,sys,urllib.request; r=json.load(sys.stdin); b=None if r['payload'] is None else json.dumps(r['payload']).encode(); q=urllib.request.Request('http://127.0.0.1:" + str(port) + "' + r['path'], data=b, method=r['method'], headers={'Content-Type':'application/json'}); x=urllib.request.urlopen(q); print(x.status); print(x.read().decode())"
    result = ssh_run(service.machine_id, shlex.join(['docker', 'exec', '-i', container, 'python', '-c', script]), input=json.dumps(request))
    lines = result.stdout.splitlines(); return (int(lines[0]), '\n'.join(lines[1:])) if lines else (0, result.stderr)
authority_status, authority_body = private_request('admin-api', 'POST', '/api/v1/admin/authority', {'uuid': AUTHORITY_ID, 'naans': [], 'fund_amount_eth': 0.05})
assert authority_status in {200, 201, 409}, authority_body
authorized_status, authorized_body = private_request('admin-api', 'POST', f'/api/v1/admin/authority/{AUTHORITY_ID}/authorize-naan', {'naan': NAAN})
assert authorized_status in {200, 409}, authorized_body
reserve = requests.post(f'{MINTER}/arks/batch', headers=HEADERS, json={'authority_id': AUTHORITY_ID, 'naan': NAAN, 'items': [{'client_item_id': PLATFORM_ITEM_ID}]}, timeout=60)
reserve.raise_for_status(); ARK = reserve.json()['results'][0]['ark']
metadata = {'title': 'Objeto de aceptación multisede', 'authors': ['Equipo dARK'], 'year': 2026, 'publisher': 'Repositorio de prueba', 'resource_type': 'article', 'language': 'es', 'abstract': 'Prueba LAN y VPN', 'subjects': ['multisede'], 'alternate_urls': [TARGET_URL]}
xml = f'<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/" xmlns:dc="http://purl.org/dc/elements/1.1/"><dc:identifier>{TARGET_URL}</dc:identifier></oai_dc:dc>'
stage = requests.put(f'{MINTER}/arks/{ARK}', headers=HEADERS, json={'authority_id': AUTHORITY_ID, 'target': TARGET_URL, 'minimal_metadata': metadata, 'original_metadata': xml, 'metadata_schema': 'oai_dc', 'metadata_media_type': 'application/xml'}, timeout=60)
stage.raise_for_status()
deadline = time.time() + POLL_TIMEOUT_SECONDS
while time.time() < deadline:
    current = requests.get(f'{MINTER}/arks/{ARK}', headers=HEADERS, timeout=30); current.raise_for_status(); body = current.json()
    if body['state'] == 'P': break
    time.sleep(3)
else: raise TimeoutError(body)
assert body.get('level1_cid') and body.get('level2_cid'), body
for cid in (body['level1_cid'], body['level2_cid']):
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    while time.time() < deadline:
        status_code, status_body = private_request('store-api', 'GET', f'/v1/status/{cid}')
        assert status_code == 200, status_body; replication = json.loads(status_body).get('replication', {})
        assert replication.get('error_replicas', 0) == 0, status_body
        if replication.get('total_replicas', 0) >= 2: break
        time.sleep(3)
    else: raise TimeoutError(f'Replication did not reach two copies: {cid}')
redirect = requests.get(f'{GATEWAY_BASE_URL}/{ARK}', allow_redirects=False, timeout=30)
assert redirect.status_code in {302, 307}
assert redirect.headers['location'] == TARGET_URL
print(f'Acceptance passed: {ARK} -> {TARGET_URL}')